# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# Kather-5K: efficient JPI revision package

This notebook implements the minimum defensible revision package without changing
the trained ResNet18, DINOv2, or UNI models. It preserves the completed 10-seed
classification and attribution-stability analyses, upgrades the completed
reference-seed faithfulness results without GPU recomputation, and confines new
robustness experiments to three prespecified seeds: **11, 89, and 181**.

New GPU work is limited to:

1. three perturbations on the existing 272-image cohort at a common 14 x 14 grid;
2. native and common 7 x 7 grid sensitivity (the common 14 x 14 result is reused);
3. a frozen-ResNet18 training-regime control;
4. a prediction-independent cohort of 128 images (16 per class);
5. a small five-versus-twenty random-deletion-repeat calibration.

Integrated Gradients and additional activation-patching designs are intentionally
out of scope. Highlighted regions indicate contribution to a model prediction, not
biological causation.

In [ ]:
from pathlib import Path
import gc
import json
import os
import shutil
import sys
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display


def locate_project_root():
    return _PUBLICATION_ROOT


def locate_completed_artifacts(project_root):
    candidates = [
        project_root / 'artifacts',
        project_root / 'Artifacts 2',
        project_root / 'artifacts 2',
    ]
    for candidate in candidates:
        if (candidate / 'dinov2_three_model').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the completed three-model artifacts')


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

COMPLETED_ROOT = locate_completed_artifacts(PROJECT_ROOT)
DATASET_DIR = (
    PROJECT_ROOT / 'Colorectal Histology MNIST'
    / 'Kather_texture_2016_image_tiles_5000'
    / 'Kather_texture_2016_image_tiles_5000'
)
ONLINE_EXTERNAL_DATASET_DIR = Path(
    'CRC-VAL-HE-7K'
)
EXTERNAL_DATASET_DIR = (
    ONLINE_EXTERNAL_DATASET_DIR
    if ONLINE_EXTERNAL_DATASET_DIR.is_dir()
    else PROJECT_ROOT / 'CRC-VAL-HE-7K'
)
GROUPED_DIR = COMPLETED_ROOT / 'grouped_oof_faithfulness'
THREE_MODEL_DIR = COMPLETED_ROOT / 'dinov2_three_model'
REVISION_DIR = PROJECT_ROOT / 'artifacts' / 'kather5k_jpi_revision'
for name in ('analysis_only', 'cohorts', 'classification', 'evaluation', 'statistics', 'figures', 'calibration'):
    (REVISION_DIR / name).mkdir(parents=True, exist_ok=True)

print('Project:', PROJECT_ROOT)
print('Completed artifacts:', COMPLETED_ROOT)
print('New outputs:', REVISION_DIR)
print('Pending external dataset:', EXTERNAL_DATASET_DIR)
print('CUDA devices:', torch.cuda.device_count())

## Prespecified compute budget

The default is the final run. Each model-stage-fold partition is written
independently, so rerunning the notebook resumes completed work. `MODEL_FILTER`
can be used to split work across two notebook kernels, for example ResNet models
on one GPU and Transformers on the other. Consolidation should be run only after
all requested model partitions are present.

In [ ]:
ALL_SEEDS = (11, 23, 41, 57, 73, 89, 101, 131, 151, 181)
SENSITIVITY_SEEDS = (11, 89, 181)  # first, middle, last; fixed before inspection
REFERENCE_SEED = 41
CALIBRATION_SEED = 89
RANDOM_COHORT_SEED = 314159

DEVICE = torch.device(
    'cuda:1' if torch.cuda.device_count() > 1
    else ('cuda:0' if torch.cuda.is_available() else 'cpu')
)
MODEL_FILTER = ('ResNet18', 'DINOv2', 'UNI', 'FrozenResNet18')
RUN_LABEL = 'all_models'
RUN_FROZEN_TRAINING = True
RUN_PERTURBATION = True
RUN_GRID = True
RUN_RANDOM_COHORT = True
RUN_FROZEN_FAITHFULNESS = True
RUN_REPEAT_CALIBRATION = True
RUN_CONSOLIDATION = True
OVERWRITE_PARTITIONS = False

RANDOM_DELETION_REPEATS = 5
CALIBRATION_REPEATS = 20
BOOTSTRAP_ITERATIONS = 5000
NUM_WORKERS = 4

print('Device:', DEVICE)
print('Sensitivity seeds:', SENSITIVITY_SEEDS)
print('Models in this run:', MODEL_FILTER)

In [ ]:
from Methods.BaselineCNN import discover_images, set_seed
from Methods.CNNBenchmark import build_resnet18
from Methods.DINOv2Attribution import build_dinov2_classifier, resolve_dinov2_transform
from Methods.GroupAwareEvaluation import run_transformer_oof, validate_oof_assignments
from Methods.KatherRevision.classification import save_classification_revision
from Methods.KatherRevision.cohorts import (
    attach_prediction_hierarchy,
    build_budgeted_cohort,
    build_random_cohort,
    explanation_targets,
    freeze_random_cohort,
)
from Methods.KatherRevision.evaluation import (
    CNNAdapter,
    TransformerAdapter,
    consolidate_partitions,
    evaluate_image_seed_partitions,
)
from Methods.KatherRevision.jpi_figures import create_jpi_figures
from Methods.KatherRevision.legacy import (
    add_scale_free_deletion_metrics,
    attach_reference_seed_analysis_families,
)
from Methods.KatherRevision.models import (
    build_frozen_resnet18_classifier,
    extract_frozen_resnet_features,
)
from Methods.KatherRevision.statistics import (
    architecture_primary_rows,
    conclusion_robustness_table,
    heterogeneity_summary,
    paired_model_inference,
    paired_top_vs_random,
    save_revision_tables,
    seed_image_source_summaries,
)
from Methods.UNIAttribution import build_uni_classifier, resolve_uni_transform

set_seed(REFERENCE_SEED)
manifest = discover_images(DATASET_DIR)
CLASS_NAMES = (
    manifest[['class_name', 'label']]
    .drop_duplicates()
    .sort_values('label')['class_name']
    .tolist()
)
assignments = pd.read_csv(GROUPED_DIR / 'folds' / 'fold_assignments.csv')
main_predictions = pd.read_csv(
    THREE_MODEL_DIR / 'classification' / 'oof_predictions_and_logits.csv'
)
selected_cohort = pd.read_csv(
    THREE_MODEL_DIR / 'three_model_faithfulness_cohort.csv'
)
validate_oof_assignments(assignments, expected_image_count=len(manifest))
for frame in (main_predictions, selected_cohort):
    frame['path'] = frame['relative_path'].map(lambda value: str(DATASET_DIR / value))
selected_cohort['cohort_name'] = 'selected_faithfulness'

assert set(main_predictions['seed']) == set(ALL_SEEDS)
assert manifest['label'].nunique() == 8
assert manifest['case_id'].nunique() == 10
print('Manifest:', len(manifest), 'images,', manifest['case_id'].nunique(), 'source groups')

# Inventory only. No CRC-VAL predictions or attributions are calculated before
# the Kather stopping rule is evaluated and the label-space protocol is frozen.
external_class_counts = pd.DataFrame(
    [
        {'external_class': directory.name, 'image_count': len(list(directory.glob('*.tif')))}
        for directory in sorted(EXTERNAL_DATASET_DIR.iterdir())
        if directory.is_dir()
    ]
)
assert set(external_class_counts['external_class']) == {
    'ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM'
}
assert external_class_counts['image_count'].sum() == 7180
external_class_counts.to_csv(
    REVISION_DIR / 'cohorts/crc_val_he_7k_inventory_pending.csv', index=False
)
display(external_class_counts)

## 1. Analysis-only corrections

This cell uses the already completed 272-image reference-seed deletion curves.
It adds normalized score AUCs, relative reductions, top-beats-random indicators,
image/source summaries, paired Wilcoxon tests, source-group hierarchical bootstrap
intervals, exact source-level sign-flip tests, and leave-one-source-out estimates.
No model is loaded.

The completed classification and stability analyses span 10 seeds; the completed
faithfulness cohort itself was evaluated at reference seed 41. These are reported
as distinct sources of uncertainty.

In [ ]:
legacy_metrics_path = THREE_MODEL_DIR / 'faithfulness' / 'three_model_faithfulness_metrics.csv'
legacy_curves_path = THREE_MODEL_DIR / 'faithfulness' / 'three_model_deletion_curves.csv'
legacy_metrics = pd.read_csv(legacy_metrics_path)
legacy_curves = pd.read_csv(legacy_curves_path)

legacy_enriched, legacy_curves_enriched = add_scale_free_deletion_metrics(
    legacy_metrics, legacy_curves
)
legacy_analysis = attach_reference_seed_analysis_families(legacy_enriched)
legacy_primary = architecture_primary_rows(legacy_analysis)

legacy_enriched.to_csv(REVISION_DIR / 'analysis_only/reference_seed_scale_free_metrics.csv', index=False)
legacy_curves_enriched.to_csv(REVISION_DIR / 'analysis_only/reference_seed_relative_deletion_curves.csv', index=False)
legacy_analysis.to_csv(REVISION_DIR / 'analysis_only/reference_seed_analysis_families.csv', index=False)

legacy_summaries = seed_image_source_summaries(legacy_primary)
for level, frame in legacy_summaries.items():
    frame.to_csv(REVISION_DIR / f'analysis_only/reference_seed_{level}_summary.csv', index=False)
legacy_tests, legacy_pairs, legacy_loso = paired_model_inference(
    legacy_primary, bootstrap_iterations=BOOTSTRAP_ITERATIONS
)
legacy_tests.to_csv(REVISION_DIR / 'analysis_only/reference_seed_paired_tests.csv', index=False)
legacy_pairs.to_csv(REVISION_DIR / 'analysis_only/reference_seed_paired_values.csv', index=False)
legacy_loso.to_csv(REVISION_DIR / 'analysis_only/reference_seed_leave_one_source_out.csv', index=False)
paired_top_vs_random(legacy_primary, BOOTSTRAP_ITERATIONS).to_csv(
    REVISION_DIR / 'analysis_only/reference_seed_top_vs_random_tests.csv', index=False
)
heterogeneity_summary(legacy_primary).to_csv(
    REVISION_DIR / 'analysis_only/reference_seed_heterogeneity.csv', index=False
)

# Preserve the completed 10-seed stability outputs as a separate optimization-
# variability analysis. They are not pooled with image/source sampling inference.
existing_stability_dir = REVISION_DIR / 'analysis_only/existing_10_seed_stability'
existing_stability_dir.mkdir(parents=True, exist_ok=True)
for source in (THREE_MODEL_DIR / 'stability').glob('*.csv'):
    shutil.copy2(source, existing_stability_dir / source.name)
display(
    legacy_primary.query(
        "analysis_family == 'primary_jointly_correct_true_class'"
    ).groupby('model').agg(
        images=('cohort_id', 'nunique'),
        spearman=('attribution_occlusion_spearman', 'mean'),
        relative_logit_auc=('top_minus_random_relative_target_logit_reduction_auc', 'mean'),
        top_beats_random=('top_beats_random_target_logit', 'mean'),
    ).round(4)
)

## 2. Freeze the 128-image prediction-independent cohort

In [ ]:
proposed_random = build_random_cohort(
    manifest, assignments, images_per_class=16, seed=RANDOM_COHORT_SEED
)
random_cohort = freeze_random_cohort(
    proposed_random,
    REVISION_DIR / 'cohorts/random_prediction_independent_128.csv',
    overwrite=False,
)
random_cohort['path'] = random_cohort['relative_path'].map(
    lambda value: str(DATASET_DIR / value)
)
assert len(random_cohort) == 128
assert random_cohort['label'].nunique() == 8
assert not random_cohort['selected_using_predictions'].any()
assert not random_cohort['selected_using_attributions'].any()
display(pd.crosstab(random_cohort['class_name'], random_cohort['case_id']))

## 3. Frozen-ResNet18 matched training control

In [ ]:
LEGACY_FROZEN_DIR = PROJECT_ROOT / 'artifacts' / 'kather5k_revision' / 'classification'
FROZEN_CHECKPOINT_DIR = REVISION_DIR / 'classification/checkpoints/frozen_resnet18'
FROZEN_CACHE_PATH = REVISION_DIR / 'classification/frozen_resnet18_features.pt'
FROZEN_PREDICTION_PATH = REVISION_DIR / 'classification/frozen_resnet18_oof_predictions.csv'

# One-time migration makes the new revision package self-contained. Once this
# cell completes, the legacy kather5k_revision directory is no longer required.
legacy_cache = LEGACY_FROZEN_DIR / 'frozen_resnet18_features.pt'
legacy_checkpoints = LEGACY_FROZEN_DIR / 'checkpoints/frozen_resnet18'
if legacy_cache.is_file() and not FROZEN_CACHE_PATH.is_file():
    FROZEN_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(legacy_cache, FROZEN_CACHE_PATH)
if legacy_checkpoints.is_dir():
    shutil.copytree(
        legacy_checkpoints,
        FROZEN_CHECKPOINT_DIR,
        dirs_exist_ok=True,
    )

frozen_predictions = (
    pd.read_csv(FROZEN_PREDICTION_PATH)
    if FROZEN_PREDICTION_PATH.is_file()
    else pd.DataFrame()
)
expected_frozen_checkpoints = [
    FROZEN_CHECKPOINT_DIR / f'seed_{seed}' / f'fold_{fold}.pt'
    for seed in ALL_SEEDS
    for fold in range(10)
]
frozen_complete = (
    not frozen_predictions.empty
    and set(frozen_predictions.get('seed', [])) == set(ALL_SEEDS)
    and frozen_predictions.groupby('seed')['relative_path'].nunique().eq(5000).all()
    and FROZEN_CACHE_PATH.is_file()
    and all(path.is_file() for path in expected_frozen_checkpoints)
)
if not frozen_complete:
    frozen_model = build_frozen_resnet18_classifier(len(CLASS_NAMES), DEVICE)
    frozen_cache = extract_frozen_resnet_features(
        frozen_model,
        manifest,
        DEVICE,
        FROZEN_CACHE_PATH,
        batch_size=128,
        num_workers=NUM_WORKERS,
    )
    if not RUN_FROZEN_TRAINING:
        raise FileNotFoundError('Frozen ResNet predictions are absent and training is disabled')
    frozen_predictions, frozen_history = run_transformer_oof(
        frozen_model,
        frozen_cache,
        manifest,
        assignments,
        CLASS_NAMES,
        DEVICE,
        FROZEN_CHECKPOINT_DIR,
        seeds=ALL_SEEDS,
        batch_size=256,
        epochs=40,
        learning_rate=1e-3,
        weight_decay=1e-4,
        patience=8,
        force_retrain=False,
        model_name='FrozenResNet18',
    )
    frozen_predictions.to_csv(FROZEN_PREDICTION_PATH, index=False)
    frozen_history.to_csv(REVISION_DIR / 'classification/frozen_resnet18_history.csv', index=False)
    del frozen_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

frozen_predictions['path'] = frozen_predictions['relative_path'].map(
    lambda value: str(DATASET_DIR / value)
)
assert set(frozen_predictions['seed']) == set(ALL_SEEDS)
assert frozen_predictions.groupby('seed')['relative_path'].nunique().eq(5000).all()
assert FROZEN_CACHE_PATH.is_file()
assert all(path.is_file() for path in expected_frozen_checkpoints)

classification_predictions = pd.concat((main_predictions, frozen_predictions), ignore_index=True)
classification_paths = save_classification_revision(
    classification_predictions,
    CLASS_NAMES,
    REVISION_DIR / 'classification',
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)
classification_per_seed = pd.read_csv(classification_paths['classification_per_seed'])
display(
    classification_per_seed.groupby('model')[['accuracy', 'balanced_accuracy', 'macro_f1']]
    .agg(['mean', 'std']).round(4)
)

## 4. Prespecified explanation targets

In [ ]:
sensitivity_predictions = main_predictions[
    main_predictions['seed'].isin(SENSITIVITY_SEEDS)
].copy()

selected_index = attach_prediction_hierarchy(selected_cohort, sensitivity_predictions)
selected_targets = explanation_targets(selected_index)
selected_primary_targets = selected_targets[
    selected_targets['analysis_family'].eq('primary_jointly_correct_true_class')
].copy()

random_index = attach_prediction_hierarchy(random_cohort, sensitivity_predictions)
random_targets = explanation_targets(random_index)
random_primary_targets = random_targets[
    random_targets['analysis_family'].eq('primary_jointly_correct_true_class')
].copy()

training_predictions = pd.concat(
    (
        sensitivity_predictions[sensitivity_predictions['model'].eq('ResNet18')],
        frozen_predictions[frozen_predictions['seed'].isin(SENSITIVITY_SEEDS)],
    ),
    ignore_index=True,
)
training_cohort = selected_cohort.copy()
training_cohort['cohort_name'] = 'selected_training_regime'
training_index = attach_prediction_hierarchy(
    training_cohort,
    training_predictions,
    model_names=('ResNet18', 'FrozenResNet18'),
)
training_targets = explanation_targets(training_index)
training_targets = training_targets[
    training_targets['analysis_family'].eq('primary_jointly_correct_true_class')
].copy()
training_targets['analysis_family'] = 'training_regime_jointly_correct_true_class'

eligible_ids = selected_index[
    selected_index['seed'].eq(CALIBRATION_SEED)
    & selected_index['all_models_correct']
]['cohort_id'].unique()
calibration_cohort = build_budgeted_cohort(
    selected_cohort[selected_cohort['cohort_id'].isin(eligible_ids)],
    images_per_class=2,
    seed=271828,
    cohort_name='repeat_calibration_16',
    preserve_stratum='cohort_stratum',
)
calibration_targets = selected_primary_targets[
    selected_primary_targets['seed'].eq(CALIBRATION_SEED)
    & selected_primary_targets['cohort_id'].isin(calibration_cohort['cohort_id'])
].copy()
calibration_targets['cohort_name'] = 'repeat_calibration_16'

for name, frame in {
    'selected_primary_targets': selected_primary_targets,
    'random_primary_targets': random_primary_targets,
    'training_regime_targets': training_targets,
    'repeat_calibration_targets': calibration_targets,
}.items():
    frame.to_csv(REVISION_DIR / f'cohorts/{name}.csv', index=False)

display(pd.DataFrame({
    'target_set': ['selected', 'random', 'training regime', 'repeat calibration'],
    'rows': [len(selected_primary_targets), len(random_primary_targets), len(training_targets), len(calibration_targets)],
    'images': [selected_primary_targets['cohort_id'].nunique(), random_primary_targets['cohort_id'].nunique(), training_targets['cohort_id'].nunique(), calibration_targets['cohort_id'].nunique()],
}))

## 5. Resumable robustness evaluation

The same deterministic random-deletion seed is derived from image, training seed,
target, perturbation, and grid. It intentionally excludes model and explanation
method, so the five random deletion sequences are shared in paired comparisons.

The 14 x 14 normalized-zero result is produced once in the perturbation stage.
The grid stage adds native and common 7 x 7 evaluations, avoiding duplicate work.
Only Grad-CAM or gradient-weighted rollout is requested; Integrated Gradients,
raw attention, ordinary rollout, and extra activation patching are not computed.

In [ ]:
METHODS = {
    'ResNet18': ('gradcam',),
    'FrozenResNet18': ('gradcam',),
    'DINOv2': ('gradient_attention_rollout',),
    'UNI': ('gradient_attention_rollout',),
}


def make_adapters(requested_models):
    active_models = set(MODEL_FILTER).intersection(requested_models)
    adapters = {}
    if 'ResNet18' in active_models:
        adapters['ResNet18'] = CNNAdapter(
            'ResNet18',
            model_builder=lambda: build_resnet18(
                num_classes=len(CLASS_NAMES), pretrained=False, freeze_backbone=False
            ),
            checkpoint_dir=THREE_MODEL_DIR / 'checkpoints/resnet18',
            device=DEVICE,
            checkpoint_type='state_dict',
            image_size=150,
            native_grid_size=5,
        )
    if 'DINOv2' in active_models:
        dino_model = build_dinov2_classifier(len(CLASS_NAMES), DEVICE)
        dino_transform, _ = resolve_dinov2_transform(dino_model.encoder)
        adapters['DINOv2'] = TransformerAdapter(
            'DINOv2', dino_model, THREE_MODEL_DIR / 'checkpoints/dinov2',
            DEVICE, dino_transform, native_grid_size=16,
        )
    if 'UNI' in active_models:
        uni_model = build_uni_classifier(
            PROJECT_ROOT,
            len(CLASS_NAMES),
            DEVICE,
            assets_dir=os.environ.get('UNI_ASSETS_DIR') or None,
        )
        uni_transform, _ = resolve_uni_transform(uni_model.encoder)
        adapters['UNI'] = TransformerAdapter(
            'UNI', uni_model, GROUPED_DIR / 'checkpoints/uni',
            DEVICE, uni_transform, native_grid_size=14,
        )
    if 'FrozenResNet18' in active_models:
        adapters['FrozenResNet18'] = CNNAdapter(
            'FrozenResNet18',
            model_builder=lambda: build_frozen_resnet18_classifier(len(CLASS_NAMES), DEVICE),
            checkpoint_dir=FROZEN_CHECKPOINT_DIR,
            device=DEVICE,
            checkpoint_type='classifier_head',
            image_size=150,
            native_grid_size=5,
        )
    return adapters


def run_stage(stage, targets, models, perturbations, common_grids, include_native, repeats):
    stage_dir = REVISION_DIR / 'evaluation' / stage
    started = time.time()
    adapters = make_adapters(models)
    try:
        for model_name in models:
            if model_name not in adapters:
                continue
            print(f'[{stage}] {model_name}')
            evaluate_image_seed_partitions(
                adapters[model_name],
                targets,
                output_dir=stage_dir,
                perturbations=perturbations,
                common_grid_sizes=common_grids,
                integrated_gradient_steps=1,
                random_repeats=repeats,
                random_null_repeats=0,
                batch_size=64,
                include_higher_res_cam=False,
                requested_methods=METHODS[model_name],
                include_native_grid=include_native,
                overwrite=OVERWRITE_PARTITIONS,
            )
            adapters[model_name].model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    finally:
        adapters.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    hours = (time.time() - started) / 3600
    print(f'{stage} elapsed hours: {hours:.2f}')
    return hours

In [ ]:
stage_times = {}
main_models = ('ResNet18', 'DINOv2', 'UNI')

if RUN_PERTURBATION:
    stage_times['perturbation'] = run_stage(
        'perturbation',
        selected_primary_targets,
        main_models,
        perturbations=('normalized_zero', 'gaussian_blur', 'local_mean'),
        common_grids=(14,),
        include_native=False,
        repeats=RANDOM_DELETION_REPEATS,
    )

if RUN_GRID:
    stage_times['grid'] = run_stage(
        'grid',
        selected_primary_targets,
        main_models,
        perturbations=('normalized_zero',),
        common_grids=(7,),
        include_native=True,
        repeats=RANDOM_DELETION_REPEATS,
    )

if RUN_RANDOM_COHORT:
    stage_times['random_cohort'] = run_stage(
        'random_cohort',
        random_primary_targets,
        main_models,
        perturbations=('normalized_zero',),
        common_grids=(14,),
        include_native=False,
        repeats=RANDOM_DELETION_REPEATS,
    )

if RUN_FROZEN_FAITHFULNESS:
    stage_times['frozen_resnet'] = run_stage(
        'frozen_resnet',
        training_targets,
        ('ResNet18', 'FrozenResNet18'),
        perturbations=('normalized_zero',),
        common_grids=(14,),
        include_native=False,
        repeats=RANDOM_DELETION_REPEATS,
    )

if RUN_REPEAT_CALIBRATION:
    stage_times['repeat_calibration'] = run_stage(
        'repeat_calibration',
        calibration_targets,
        main_models,
        perturbations=('normalized_zero',),
        common_grids=(14,),
        include_native=False,
        repeats=CALIBRATION_REPEATS,
    )

pd.DataFrame(
    [{'stage': name, 'elapsed_hours': value} for name, value in stage_times.items()]
).to_csv(
    REVISION_DIR / f'evaluation/stage_runtime_hours_{RUN_LABEL}.csv', index=False
)

## 6. Consolidate, infer, and apply the stopping rule

In [ ]:
def consolidate_stage(stage):
    stage_dir = REVISION_DIR / 'evaluation' / stage
    output_dir = stage_dir / 'consolidated'
    output_dir.mkdir(parents=True, exist_ok=True)
    outputs = {}
    for artifact, filename in (
        ('metrics', 'image_seed_metrics.csv'),
        ('deletion_curves', 'deletion_curves.csv'),
        ('attribution_maps', 'attribution_maps.csv'),
        ('occlusion_scores', 'occlusion_scores.csv'),
    ):
        frame = consolidate_partitions(stage_dir, artifact, output_dir / filename)
        frame['revision_stage'] = stage
        frame.to_csv(output_dir / filename, index=False)
        outputs[artifact] = frame
    return outputs


if RUN_CONSOLIDATION:
    required_stages = ('perturbation', 'grid', 'random_cohort', 'frozen_resnet', 'repeat_calibration')
    consolidated = {stage: consolidate_stage(stage) for stage in required_stages}
    main_stages = ('perturbation', 'grid', 'random_cohort', 'frozen_resnet')
    metrics = pd.concat([consolidated[stage]['metrics'] for stage in main_stages], ignore_index=True)
    curves = pd.concat([consolidated[stage]['deletion_curves'] for stage in main_stages], ignore_index=True)
    maps = pd.concat([consolidated[stage]['attribution_maps'] for stage in main_stages], ignore_index=True)
    occlusion = pd.concat([consolidated[stage]['occlusion_scores'] for stage in main_stages], ignore_index=True)

    combined_dir = REVISION_DIR / 'evaluation/consolidated'
    combined_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(combined_dir / 'image_seed_metrics.csv', index=False)
    curves.to_csv(combined_dir / 'deletion_curves.csv', index=False)
    maps.to_csv(combined_dir / 'attribution_maps.csv', index=False)
    occlusion.to_csv(combined_dir / 'occlusion_scores.csv', index=False)

    assert set(metrics['seed']) == set(SENSITIVITY_SEEDS)
    assert not metrics['method'].eq('integrated_gradients').any()
    assert {'normalized_zero', 'gaussian_blur', 'local_mean'} <= set(metrics['perturbation'])
    assert {'common_14', 'common_7'} <= set(metrics['grid_label'])

    table_paths = save_revision_tables(
        metrics,
        REVISION_DIR / 'statistics',
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    )

    training_metrics = metrics[
        metrics['analysis_family'].eq('training_regime_jointly_correct_true_class')
    ]
    training_tests, training_pairs, training_loso = paired_model_inference(
        training_metrics,
        model_pairs=(('FrozenResNet18', 'ResNet18'),),
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    )
    training_tests.to_csv(REVISION_DIR / 'statistics/frozen_resnet_paired_tests.csv', index=False)
    training_pairs.to_csv(REVISION_DIR / 'statistics/frozen_resnet_paired_values.csv', index=False)
    training_loso.to_csv(REVISION_DIR / 'statistics/frozen_resnet_leave_one_source_out.csv', index=False)

    calibration20 = architecture_primary_rows(consolidated['repeat_calibration']['metrics'])
    calibration20 = calibration20[
        calibration20['perturbation'].eq('normalized_zero')
        & calibration20['grid_label'].eq('common_14')
    ][['cohort_id', 'model', 'random_relative_target_logit_reduction_auc']].rename(
        columns={'random_relative_target_logit_reduction_auc': 'twenty_repeat_auc'}
    )
    calibration5 = architecture_primary_rows(consolidated['perturbation']['metrics'])
    calibration5 = calibration5[
        calibration5['seed'].eq(CALIBRATION_SEED)
        & calibration5['cohort_id'].isin(calibration_cohort['cohort_id'])
        & calibration5['perturbation'].eq('normalized_zero')
        & calibration5['grid_label'].eq('common_14')
    ][['cohort_id', 'model', 'random_relative_target_logit_reduction_auc']].rename(
        columns={'random_relative_target_logit_reduction_auc': 'five_repeat_auc'}
    )
    repeat_calibration = calibration5.merge(
        calibration20, on=['cohort_id', 'model'], validate='one_to_one'
    )
    repeat_calibration['absolute_difference'] = (
        repeat_calibration['five_repeat_auc'] - repeat_calibration['twenty_repeat_auc']
    ).abs()
    repeat_calibration.to_csv(REVISION_DIR / 'calibration/random_repeat_paired_values.csv', index=False)
    repeat_summary = repeat_calibration.groupby('model').agg(
        images=('cohort_id', 'nunique'),
        mean_absolute_difference=('absolute_difference', 'mean'),
        spearman=('five_repeat_auc', lambda values: values.corr(
            repeat_calibration.loc[values.index, 'twenty_repeat_auc'], method='spearman'
        )),
    ).reset_index()
    repeat_summary.to_csv(REVISION_DIR / 'calibration/random_repeat_summary.csv', index=False)

    robustness = conclusion_robustness_table(metrics)
    robustness['decision_rule'] = (
        'Stop at 3 seeds when direction is consistent; expand only a contradictory axis to 10 seeds.'
    )
    robustness.to_csv(REVISION_DIR / 'statistics/final_conclusion_robustness.csv', index=False)
    display(robustness)
    display(repeat_summary)

## 7. Publication figures and run manifest

In [ ]:
if RUN_CONSOLIDATION:
    figure_paths = create_jpi_figures(
        metrics,
        classification_per_seed,
        repeat_calibration,
        REVISION_DIR / 'figures',
    )
    for name, path in figure_paths.items():
        print(name, '->', path)

    # Keep the already completed stability and representative-map figures in the
    # same submission-ready folder without recomputing them.
    existing_figure_dir = COMPLETED_ROOT / 'final_three_model_comparison' / 'figures'
    for filename in (
        '05_cross_seed_stability.png',
        '07_representative_correct_attribution_maps.png',
        '08_representative_incorrect_attribution_maps.png',
    ):
        source = existing_figure_dir / filename
        if source.is_file():
            shutil.copy2(source, REVISION_DIR / 'figures' / f'existing_{filename}')

run_manifest = {
    'dataset': 'Kather-5K',
    'source_groups': 10,
    'classification_and_existing_stability_seeds': list(ALL_SEEDS),
    'new_sensitivity_seeds': list(SENSITIVITY_SEEDS),
    'new_sensitivity_seed_role': 'optimization sensitivity; not biological replicates',
    'perturbations': ['normalized_zero', 'gaussian_blur', 'local_mean'],
    'grids': ['native', 'common_14', 'common_7'],
    'random_cohort_images': 128,
    'random_deletion_repeats': RANDOM_DELETION_REPEATS,
    'random_repeat_calibration': [5, 20],
    'integrated_gradients_run': False,
    'frozen_resnet_control_run': RUN_FROZEN_TRAINING,
    'activation_patching_expanded': False,
    'external_validation_started': False,
    'external_validation_path': str(EXTERNAL_DATASET_DIR),
    'external_validation_images': 7180,
    'external_validation_classes': 9,
    'external_validation_status': (
        'pending Kather stopping rule and a frozen 8-to-9-class harmonization protocol'
    ),
    'stopping_rule': (
        'If all three-seed sensitivity directions agree with the primary result, stop. '
        'If one axis materially reverses the ranking, expand only that axis to 10 seeds.'
    ),
    'interpretation': 'attribution marks model-prediction contribution, not biological causation',
}
(REVISION_DIR / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2))
print('Complete:', REVISION_DIR)

## Manuscript decision rule

The existing 10-seed classification and stability analyses remain the main evidence
for optimization variability. New perturbation, grid, random-cohort, and training-
regime results are explicitly sensitivity analyses based on three prespecified
seeds. Source group is the primary sampling cluster throughout.

If the robustness table is directionally consistent, stop Kather-5K computation
and revise the manuscript. If one sensitivity materially reverses the ranking,
expand only that sensitivity to all 10 seeds. Do not automatically add perturbation
types, Integrated Gradients, or further activation-patching experiments.

Use the conservative conclusion only when supported:

> UNI + gradient-weighted rollout was the most faithful among the tested
> model-training-explanation pipelines under the primary evaluation, while
> absolute attribution-occlusion agreement remained modest.

Do not attribute the difference specifically to pathology pretraining without a
matched architecture/pretraining experiment, and do not make biological-causality
claims.